In [1]:
# -*- coding: utf-8 -*-
"""
Created on Thu Mar 19 13:20:22 2026

@author: csheehan1
"""

#%%

## Code Block 1
import numpy as np
from matplotlib import pyplot as plt
# from tqdm.notebook import tqdm

from landlab.components import FlowAccumulator, OverlandFlow, PriorityFloodFlowRouter
from landlab.io import esri_ascii, read_esri_ascii
from landlab import RasterModelGrid, imshow_grid

#%%

## Code Block 2

basin_flag = "Long"  # 'Square' or Long'
storm_flag = "LongerDuration"  # 'Base' or'HigherIntensity' or 'LongerDuration'

## If the basin flag matches one of the two select basins,
## below will set the filename which to read the DEM from and
## the outlet link and upstream link to sample discharge values
## from for plotting.

if basin_flag == "Square":
    watershed_dem = "Square_TestBasin.asc"
    ## Reading in the DEM given the filename from above
    with open(watershed_dem) as fp:
        rmg = esri_ascii.load(fp, name="topographic__elevation", at="node")
    outlet_node_to_sample = 300
    outlet_link_to_sample = rmg.links_at_node[outlet_node_to_sample][3]
    upstream_node_to_sample = 28689
    upstream_link_to_sample = rmg.links_at_node[upstream_node_to_sample][3]
    midstream_node_to_sample = 9102
    midstream_link_to_sample = rmg.links_at_node[midstream_node_to_sample][3]
##else:
    ##watershed_dem = "Long_TestBasin.asc"
    ## Reading in the DEM given the filename from above
   ## with open(watershed_dem) as fp:
   ##     rmg = esri_ascii.load(fp, name="topographic__elevation", at="node")
   ## outlet_node_to_sample = 150
   ## outlet_link_to_sample = rmg.links_at_node[outlet_node_to_sample][3]
    ##upstream_node_to_sample = 33859
   ## upstream_link_to_sample = rmg.links_at_node[upstream_node_to_sample][3]
   ## midstream_node_to_sample = 14658
   ## midstream_link_to_sample = rmg.links_at_node[midstream_node_to_sample][2]
else: 
    watershed_dem = "saucon_ws_4_15.asc"
    ## Reading in the DEM given the filename from above
    with open(watershed_dem) as fp:
        (rmg, zr) = read_esri_ascii(fp, name="topographic__elevation")
        
    outlet_node_to_sample = 39271
    outlet_link_to_sample = rmg.links_at_node[outlet_node_to_sample][3]
    upstream_node_to_sample = 9099
    upstream_link_to_sample = rmg.links_at_node[upstream_node_to_sample][3]
    midstream_node_to_sample = 24866
    midstream_link_to_sample = rmg.links_at_node[midstream_node_to_sample][3]
    # with open(watershed_dem) as fp:
    #     info, data = esri_ascii.lazy_load(fp, name="topographic__elevation", at="node")


# Define outlet node
ramp_nodes = [38807, 39250, 39471, 39692, 39913, 40134, 40355, 40576, 40797, 41018, 41239, 41460, 41681, 41902, 42123, 42344, 42565, 42786, 43007, 43228, 43449, 43670, 43891, 44112, 44333, 44554, 44775, 44996, 45217]
outlet_node = 45217

# Turn on new outlet node
rmg.at_node['topographic__elevation'][ramp_nodes] = 69.7

## Set boundary coditions on the grid
#rmg.set_watershed_boundary_condition(rmg.at_node["topographic__elevation"])
rmg.set_nodata_nodes_to_closed(rmg.at_node["topographic__elevation"], -9999.0)
rmg.set_status_at_node_on_edges(right=4, top=4, left=4, bottom=4)
rmg.status_at_node[outlet_node] = rmg.BC_NODE_IS_FIXED_VALUE
#rmg.status_at_node[outlet_node] = rmg.BC_NODE_IS_FIXED_VALUE

## The Flow Router calculates drainage area, which is helpful for
## calculating equilibrium discharge, which we illustrate later.
fr = PriorityFloodFlowRouter(rmg)
#fr = FlowAccumulator(rmg)  # Instantiate flow router
fr.run_one_step()  # Drainage area calculated

%matplotlib qt
plt.ion()
imshow_grid(rmg, 'drainage_area')
plt.plot(rmg.x_of_node[outlet_node], rmg.y_of_node[outlet_node], 'go')

#%%

da = rmg.at_node["drainage_area"]
da_sorted = np.sort(da)[::-1]
Station_da = da_sorted[1]
Outlet_da = da_sorted[0]
Station_node = np.where(da == Station_da)
Outlet_node = np.where(da == Outlet_da)
Station_node = Station_node[0]
Outlet_node = Outlet_node[0]
#gauge_node = np.where(da_sorted == da_sorted[1])
# gauge_node
#gauge_node = np.where(da_sorted == da_sorted[-2])
#gauge_node
print(Station_node)
print(Outlet_node)
rmg.imshow("drainage_area")

#%%

## Code Block 3
## instantiate OverlandFlow object
rmg.add_zeros("surface_water__depth", at="node")
of = OverlandFlow(rmg, alpha=0.45, steep_slopes=True)

## HelloAssign storm conditions based on flag in Code Block 2
if storm_flag == "Base":
    starting_precip_mmhr = 5.0
    starting_precip_ms = starting_precip_mmhr * (2.77778 * 10**-7)
    storm_duration = 7200.0
elif storm_flag == "HigherIntensity":
    starting_precip_mmhr = 10.0
    starting_precip_ms = starting_precip_mmhr * (2.77778 * 10**-7)
    storm_duration = 3600.0
elif storm_flag == "LongerDuration":
    starting_precip_mmhr = 2.5
    starting_precip_ms = starting_precip_mmhr * (2.77778 * 10**-7)
    storm_duration = 14400.0
    
#%%

## Code Block 4

plt.figure(1)
# imshow_grid(rmg, z)  # plot the DEM
rmg.imshow("topographic__elevation", limits=(50, 300))
#rmg.at_node["topographic__elevation"][39028]= -9999.0
plt.plot(rmg.node_x[outlet_node_to_sample], rmg.node_y[outlet_node_to_sample], "yo")
plt.plot(rmg.node_x[upstream_node_to_sample], rmg.node_y[upstream_node_to_sample], "bo")
plt.plot(
    rmg.node_x[midstream_node_to_sample], rmg.node_y[midstream_node_to_sample], "go"
)

#%%

## Code Block 5

elapsed_time = 1.0  # s
model_run_time = 43200.0  # s

## Lists for saving data
discharge_at_outlet = []
discharge_upstream = []
discharge_midstream = []
hydrograph_time = []

## Setting initial fields...
rmg.at_node["surface_water__discharge"] = np.zeros(rmg.number_of_nodes)

#%%

## Code Block 6
#SPACE
from landlab.components import SpaceLargeScaleEroder

space = SpaceLargeScaleEroder(
    rmg,
    K_sed=5e-6,
    K_br=1e-6,
    F_f=0.4,
    phi=0.4,
    H_star=1.0,
    v_s=0.001,
    m_sp=0.5,
    n_sp=1.0,
)

# create a second model grid that is a copy of the first
# and run it on that

# with tqdm(total=model_run_time) as pbar:
while elapsed_time < model_run_time:
    # Setting the adaptive time step
    of.dt = of.calc_time_step()

    ## The storm starts when the model starts. While the elapsed time is less
    ## than the storm duration, we add water to the system as rainfall.
    if elapsed_time < (storm_duration):
        of.rainfall_intensity = starting_precip_ms
    else:  # elapsed time exceeds the storm duration, rainfall ceases.
        of.rainfall_intensity = 0.0
    
    of.run_one_step()  # Generating overland flow based on the deAlmeida solution.
    space.run_one_step(of.dt)#erode and calc sediment flux
    test == 0
    ## Append time and discharge to their lists to save data and for plotting.
    hydrograph_time.append(elapsed_time)
    q = rmg.at_link["surface_water__discharge"]
    discharge_at_outlet.append(np.abs(q[outlet_link_to_sample]) * rmg.dx)
    discharge_upstream.append(np.abs(q[upstream_link_to_sample]) * rmg.dx)
    discharge_midstream.append(np.abs(q[midstream_link_to_sample]) * rmg.dx)

    elapsed_time += of.dt

    # pbar.update(of.dt)

#%%

## Code Block 7

## Calculate equilibrium discharge at each point for reference
outlet_eq_q = starting_precip_ms * rmg.at_node["drainage_area"][outlet_node_to_sample]
midstream_eq_q = (
    starting_precip_ms * rmg.at_node["drainage_area"][midstream_node_to_sample]
)
upstream_eq_q = (
    starting_precip_ms * rmg.at_node["drainage_area"][upstream_node_to_sample]
)


## Plotting hydrographs and equilibrium discharge
plt.figure(2)
plt.plot(hydrograph_time, discharge_at_outlet, "y-", label="outlet")
plt.plot(
    [np.min(hydrograph_time), np.max(hydrograph_time)],
    [outlet_eq_q, outlet_eq_q],
    "y--",
    label="outlet eq Q",
)
plt.plot(hydrograph_time, discharge_midstream, "g-", label="midstream")
plt.plot(
    [np.min(hydrograph_time), np.max(hydrograph_time)],
    [midstream_eq_q, midstream_eq_q],
    "g--",
    label="midstream eq Q",
)
plt.plot(hydrograph_time, discharge_upstream, "b-", label="upstream")
plt.plot(
    [np.min(hydrograph_time), np.max(hydrograph_time)],
    [upstream_eq_q, upstream_eq_q],
    "b--",
    label="upstream eq Q",
)

## Plot storm end and center of storm for reference
plt.plot(
    [storm_duration, storm_duration], [0, 100], "k-", linewidth=2, label="storm end"
)
plt.plot(
    [storm_duration / 2, storm_duration / 2], [0, 100], "k:", label="storm mid point"
)

plt.ylabel("Discharge (cms)")
plt.xlabel("Time (seconds)")
plt.legend(loc="upper right")
title_text = "Hydrographs, Storm is " + storm_flag + ", Watershed is " + basin_flag
plt.title(title_text)
plt.axis([0, np.max(hydrograph_time), 0, 100])

#%%

rmg.imshow("drainage_area")
#rmg.imshow("surface_water__depth")
plt.plot(rmg.node_x[outlet_node_to_sample], rmg.node_y[outlet_node_to_sample], "yo")

#%%

da = rmg.at_node["drainage_area"]
da_sorted = np.sort(da)[::-1]
Station_da = da_sorted[1]
Station_node = np.where(da == Station_da)
Station_node = Station_node[0]
#gauge_node = np.where(da_sorted == da_sorted[1])
# gauge_node
#gauge_node = np.where(da_sorted == da_sorted[-2])
#gauge_node
Station_node









/tmp/ipykernel_22873/3551571781.py:57: DeprecationWarning: landlab.io.read_asc_header has been deprecated, use landlab.io.esri_ascii.parse instead
  (rmg, zr) = read_esri_ascii(fp, name="topographic__elevation")


[44996 45217]
[44996 45217]


FieldError: SpaceLargeScaleEroder is missing required input field: soil__depth at node